# ROCKET Baseline — KeFRA RKE Replay (from `bmp_graph_to_csv.py` output)

Consumes the CSVs produced by your BMP converter (one row per image: `file, amp_0000, amp_0001, …`).
Same ROCKET-family pipeline as the NFC baseline, adapted for the 3 KeFRA signal classes.

**Why this is *not* the LOTO notebook**

LOTO was needed for NFC because the same four physical tags appear in both normal and relay classes, so a
random split leaks tag identity. KeFRA has no per-device grouping to hold out — it's three signal-type
classes (real / fake-high / fake-low). So the honest evaluation here is a stratified k-fold over the 340
samples, not LOTO.

**Getting labels in**

The converter writes `file` + `amp_*` but no class. Run it once per class folder, then tag each CSV on
load:
```
python bmp_graph_to_csv.py "Real_Signal/*.bmp"           --out real.csv      --points 512
python bmp_graph_to_csv.py "Fake_Signal_High_Gain/*.bmp" --out fake_high.csv --points 512
python bmp_graph_to_csv.py "Fake_Signal_Low_Gain/*.bmp"  --out fake_low.csv  --points 512
```
Using `--points 512` keeps every row the same length regardless of image width.

## 1. Setup

In [ ]:
# Colab: %pip install aeon scikit-learn numpy pandas matplotlib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.linear_model import RidgeClassifierCV
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, ConfusionMatrixDisplay

USE_MINIROCKET = True
if USE_MINIROCKET:
    from aeon.transformations.collection.convolution_based import MiniRocket as RocketTransform
else:
    from aeon.transformations.collection.convolution_based import Rocket as RocketTransform
print("Transform:", RocketTransform.__name__)

## 2. Config

Map each per-class CSV to its label. `NORMAL_CLASSES` defines the binary (real vs fake) collapse.

In [ ]:
LABEL_CSVS = {
    "real.csv":      "real",
    "fake_high.csv": "fake_high",
    "fake_low.csv":  "fake_low",
}
NORMAL_CLASSES = ["real"]
CLASSES        = ["real", "fake_high", "fake_low"]

N_SPLITS     = 5
RANDOM_STATE = 42

## 3. Load + label

Reads each CSV, drops the non-numeric `file` column, keeps the `amp_*` columns as the signal, and assigns
the label from which file the row came from.

In [ ]:
frames = []
for path, label in LABEL_CSVS.items():
    d = pd.read_csv(path)
    d["label"] = label
    frames.append(d)
df = pd.concat(frames, ignore_index=True)

amp_cols = [c for c in df.columns if c.startswith("amp_")]
amp_cols = sorted(amp_cols, key=lambda c: int(c.split("_")[1]))
assert amp_cols, "No amp_* columns found — did the converter run with the default column names?"

print(f"{len(df)} samples, {len(amp_cols)} amplitude points each")
print(df["label"].value_counts())

X_raw = df[amp_cols].to_numpy(np.float32)
# per-sample z-normalize (amplitudes are in dB; z-norm removes per-image offset/scale)
X_raw = (X_raw - X_raw.mean(1, keepdims=True)) / (X_raw.std(1, keepdims=True) + 1e-8)
X = X_raw[:, np.newaxis, :]          # (N, 1, n_points)
y = df["label"].to_numpy()

## 4. Sanity check — one trace per class

If real vs fake curves look distinguishable, the converter preserved the signal. If they're identical, the
extraction (e.g. `--method`, `--threshold`, or the y-axis scale) needs revisiting before trusting results.

In [ ]:
fig, ax = plt.subplots(figsize=(12, 4))
for cls in CLASSES:
    i = np.where(y == cls)[0][0]
    ax.plot(X[i, 0], lw=1, label=cls)
ax.set_title("One extracted trace per class"); ax.set_xlabel("amplitude point (resampled)")
ax.legend(); plt.tight_layout(); plt.show()

## 5. ROCKET + ridge, stratified 5-fold

With only ~340 samples a single split is noisy, so we collect out-of-fold predictions across 5 folds. The
random kernels are fit on each training fold only.

In [ ]:
skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)
oof = np.empty_like(y); fold_acc = []

for k, (tr, te) in enumerate(skf.split(np.zeros(len(y)), y), 1):
    rk = RocketTransform(random_state=RANDOM_STATE)
    Ftr = np.asarray(rk.fit_transform(X[tr])); Fte = np.asarray(rk.transform(X[te]))
    mu, sd = Ftr.mean(0), Ftr.std(0) + 1e-8
    clf = RidgeClassifierCV(alphas=np.logspace(-3, 3, 10))
    clf.fit((Ftr - mu) / sd, y[tr])
    oof[te] = clf.predict((Fte - mu) / sd)
    fold_acc.append(accuracy_score(y[te], oof[te]))
    print(f"fold {k}: acc = {fold_acc[-1]*100:.2f}%")

print(f"\nMean CV accuracy: {np.mean(fold_acc)*100:.2f}% (+/- {np.std(fold_acc)*100:.2f})")

## 6. Results — 3-class

In [ ]:
print(f"OOF accuracy: {accuracy_score(y, oof)*100:.2f}%\n")
print(classification_report(y, oof, labels=CLASSES, digits=4, zero_division=0))

cm = confusion_matrix(y, oof, labels=CLASSES)
fig, ax = plt.subplots(figsize=(6, 5))
ConfusionMatrixDisplay(cm, display_labels=CLASSES).plot(ax=ax, xticks_rotation=45, colorbar=True)
ax.set_title(f"ROCKET 3-class (OOF acc {accuracy_score(y, oof)*100:.2f}%)")
plt.tight_layout(); plt.show()

## 7. Results — binary (real vs fake)

In [ ]:
to_bin = lambda a: np.where(np.isin(a, NORMAL_CLASSES), "real", "fake")
yb, pb = to_bin(y), to_bin(oof)
print(f"Binary OOF accuracy: {accuracy_score(yb, pb)*100:.2f}%\n")
print(classification_report(yb, pb, labels=["real", "fake"], digits=4, zero_division=0))

## 8. Notes

- **Compare to the paper.** Abu Al-Haija & Alsulami reported 99.71% with ResNet50 on the raw images. This
  pipeline traces each image down to a 1D curve first, discarding the second dimension — so expect ROCKET
  to trail. The gap quantifies what the image-native model gains from the full 2D representation.
- **Watch the extraction knobs.** `--method top` follows peaks; `--method median`/`center` follows a
  centerline. If the sanity plot looks wrong, that's the first thing to change, then `--threshold`.
- **A leakage caveat still applies in spirit.** KeFRA exposes no capture/session ID, so a grouped split
  isn't possible — but if multiple images came from the same recording, even stratified k-fold can be mildly
  optimistic. Worth a sentence in the write-up, echoing the NFC tag-leakage lesson.